In [20]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [21]:
SEED = 42
np.random.seed(SEED)


In [22]:
# =========================================================
# 1) TEMPERATURE REGRESSION
# =========================================================
exp_temp_encoding = Experiment(
    name="temperature_regression_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="temperature",
        target_mode="regression",
        # target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=[
            ("temperature",),
            # ("temperature", "humidity"),
            # ("temperature", "humidity", "pressure"),
            # ("temperature", "humidity", "pressure", "wind_speed"),
            # ("temperature", "humidity", "pressure", "wind_speed", "wind_direction"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=[
            # ("Vancouver",),
            ("Jerusalem",),
            # ("Vancouver", "Seattle", "Portland"),
            # ("Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem")
        ]
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="regression",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers = [
            (8, 8),
            (32, 16),
            (32, 64),
            # (128, 64),
        ],
        loss="huber",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=500,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)


In [23]:
search = Search()

results1 = search.run(exp_temp_encoding)



Starting experiment: temperature_regression_encoding

Building dataset
  → TRAIN split


Jerusalem | windows: 100%|██████████| 1518/1518 [00:02<00:00, 662.67it/s]


  → TEST split


Jerusalem | windows: 100%|██████████| 361/361 [00:00<00:00, 719.53it/s]



Configuration run 1/3:
WEATHER (variable):
  - input_variables: ('temperature',)
  - cities: ('Jerusalem',)
MLP (variable):
  - hidden_layers: (8, 8)

Training model


...

Training finished in 3.88 seconds

Configuration run 2/3:
WEATHER (variable):
  - input_variables: ('temperature',)
  - cities: ('Jerusalem',)
MLP (variable):
  - hidden_layers: (32, 16)

Training model


Training: 100%|██████████| 400/400 [00:05<00:00, 68.71it/s, acc=n/a, loss=1.3870, lr=0.000181319]


Training finished in 5.82 seconds

Configuration run 3/3:
WEATHER (variable):
  - input_variables: ('temperature',)
  - cities: ('Jerusalem',)
MLP (variable):
  - hidden_layers: (32, 64)

Training model


Training: 100%|██████████| 400/400 [00:08<00:00, 45.38it/s, acc=n/a, loss=1.3442, lr=0.000181319]

Training finished in 8.82 seconds

Experiment finished | total runs = 3



In [24]:
from IPython.core.display import HTML

for run in results1:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    if model.task == "binary":
        raise ValueError("Expected regression task, got binary.")

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    metrics = regression_report(
        y_true=y_test,
        y_pred=y_pred,
    )

    acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
    acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

    # if acc_2 <= 0.69:
    #     continue

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================


    if acc_2 >= 0.8:
        color = "#2e7d32"   # dark green
    elif acc_2 >= 0.75:
        color = "#558b2f"   # olive green
    elif acc_2 >= 0.70:
        color = "#f9a825"   # amber
    elif acc_2 >= 0.65:
        color = "#ef6c00"   # orange
    else:
        color = "#c62828"   # red

    print("=== TEST METRICS (REGRESSION) ===")
    print(f"MAE              : {metrics['mae']:.4f}")
    print(f"MSE              : {metrics['mse']:.4f}")
    print(f"RMSE             : {metrics['rmse']:.4f}")
    display(HTML(
        f"""
        <div style="
            font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
            font-size: 13px;
            color: {color};
            padding-left: 12px;
            margin: 4px 0;
        ">
            <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
        </div>
        """
    ))
    print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
    - cities: ('Jerusalem',)
  MLP:
    - hidden_layers: (8, 8)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 8.2446
MSE              : 1501.6807
RMSE             : 38.7515


Accuracy |err|≤2°C   : 0.4177


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
    - cities: ('Jerusalem',)
  MLP:
    - hidden_layers: (32, 16)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.5793
MSE              : 4.5098
RMSE             : 2.1236


Accuracy |err|≤2°C   : 0.7591


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
    - cities: ('Jerusalem',)
  MLP:
    - hidden_layers: (32, 64)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.3374
MSE              : 3.4692
RMSE             : 1.8626


Accuracy |err|≤2°C   : 0.8018
